Project 1 — Finding Similar Scientific Abstracts in arXiv

The goal of this project is to identify pairs of similar scientific paper abstracts from the arXiv dataset.

The adopted pipeline is:
1. dataset download and loading
2. text preprocessing
3. shingle-based representation
4. MinHash signature construction
5. LSH-based candidate generation
6. exact Jaccard similarity filtering
7. analysis of the final similar pairs

In [2]:
import os
import json
import zipfile
import re
import pandas as pd
!pip install -q datasketch
from datasketch import MinHash, MinHashLSH

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 4.0 MB/s eta 0:00:00


In this section, we define the main parameters of the experiment:
- sample size
- minimum abstract length
- shingle size
- MinHash signature length
- LSH threshold
- final Jaccard threshold

In [3]:
SUBSAMPLE_SIZE = 100000
RANDOM_SEED = 42

MIN_TOKENS = 20
SHINGLE_SIZE = 3

num_perm = 64
threshold = 0.5
final_threshold = 0.5
high_threshold = 0.6

In [4]:
# Download the arXiv dataset from Kaggle
os.environ["KAGGLE_USERNAME"] = "xxxxxx"
os.environ["KAGGLE_KEY"] = "xxxxx"

DATASET_LINK = "Cornell-University/arxiv"
DOWNLOAD_DIR = "/content"
EXTRACT_DIR = "/content/data"

!pip install -q kaggle
!kaggle datasets download -d {DATASET_LINK} -p {DOWNLOAD_DIR}

os.makedirs(EXTRACT_DIR, exist_ok=True)

zip_files = [f for f in os.listdir(DOWNLOAD_DIR) if f.endswith(".zip")]

for zip_file in zip_files:
    zip_path = os.path.join(DOWNLOAD_DIR, zip_file)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EXTRACT_DIR)

json_path = "/content/data/arxiv-metadata-oai-snapshot.json"

print("Downloaded zip files:", zip_files)
print("Extracted files:", os.listdir(EXTRACT_DIR))

Dataset URL: https://www.kaggle.com/datasets/Cornell-University/arxiv
License(s): CC0-1.0
100% 1.59G/1.59G [00:42<00:00, 40.2MB/s]

Downloaded zip files: ['arxiv.zip']
Extracted files: ['arxiv-metadata-oai-snapshot.json']


In [5]:
# Keep only the fields needed for the project: id, title, abstract
records = []

with open(json_path, "r", encoding="utf-8") as f:
    for line in f:
        x = json.loads(line)
        records.append([x["id"], x["title"], x["abstract"]])

df = pd.DataFrame(records, columns=["id", "title", "abstract"])

print("Rows in full dataset:", len(df))
print("Columns:", df.columns.tolist())
df.head()

Rows in full dataset: 2982054
Columns: ['id', 'title', 'abstract']


,id,title,abstract
0,0704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-..."
2,0704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...
3,0704.0004,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...


### Subsampling and abstract cleaning

To make the experiment feasible in Colab, we work on a subsample of the dataset.
We also remove rows with missing or empty abstracts.

In [6]:
df_sample = df.sample(n=SUBSAMPLE_SIZE, random_state=RANDOM_SEED).copy()

df_sample = df_sample.dropna(subset=["abstract"])
df_sample = df_sample[df_sample["abstract"].str.strip() != ""].copy()

print("Cleaned and Sampled Rows", len(df_sample))

Cleaned and Sampled Rows 100000


In [7]:
stop_words = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "for",
    "with", "is", "are", "was", "were", "be", "by", "this", "that",
    "these", "those", "we", "our", "their", "it", "as", "at", "from",
    "can", "may", "using"
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = text.strip()
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return words

df_sample["tokens"] = df_sample["abstract"].apply(clean_text)
df_sample["num_tokens"] = df_sample["tokens"].apply(len)

df_sample = df_sample[df_sample["num_tokens"] >= MIN_TOKENS].copy()

print("Processed Rows", len(df_sample))
df_sample[["abstract", "tokens", "num_tokens"]].head()

Processed Rows 98618


,abstract,tokens,num_tokens
1790678,"Recently, vision transformer (ViT) based mul...","[recently, vision, transformer, vit, based, mu...",210
92487,We report on the development of a new theore...,"[report, development, new, theoretical, tool, ...",55
2423472,"In this paper, we investigate the distributed ...","[paper, investigate, distributed, state, estim...",146
2537267,LLM post-training involves many diverse datase...,"[llm, post, training, involves, many, diverse,...",124
176785,This dissertation is devoted to the resoluti...,"[dissertation, devoted, resolution, plateau, p...",160


In [8]:
# Each abstract is represented with word shingles

def make_shingles(words, k=SHINGLE_SIZE):
    shingles = []
    for i in range(len(words) - k + 1):
        shingle = " ".join(words[i:i+k])
        shingles.append(shingle)
    return shingles

df_sample["shingles"] = df_sample["tokens"].apply(make_shingles)
df_sample["shingle_set"] = df_sample["shingles"].apply(set)

df_sample[["tokens", "shingles"]].head()

,tokens,shingles
1790678,"[recently, vision, transformer, vit, based, mu...","[recently vision transformer, vision transform..."
92487,"[report, development, new, theoretical, tool, ...","[report development new, development new theor..."
2423472,"[paper, investigate, distributed, state, estim...","[paper investigate distributed, investigate di..."
2537267,"[llm, post, training, involves, many, diverse,...","[llm post training, post training involves, tr..."
176785,"[dissertation, devoted, resolution, plateau, p...","[dissertation devoted resolution, devoted reso..."


In [9]:
# One example of the transformation to inspect tokens and shingles

print("TITLE:")
print(df_sample.iloc[0]["title"])

print("\nFIRST 10 TOKENS:")
print(df_sample.iloc[0]["tokens"][:10])

print("\nFIRST 5 SHINGLES:")
print(df_sample.iloc[0]["shingles"][:5])

TITLE:
Rethinking Vision Transformer and Masked Autoencoder in Multimodal Face
  Anti-Spoofing

FIRST 10 TOKENS:
['recently', 'vision', 'transformer', 'vit', 'based', 'multimodal', 'learning', 'methods', 'have', 'been']

FIRST 5 SHINGLES:
['recently vision transformer', 'vision transformer vit', 'transformer vit based', 'vit based multimodal', 'based multimodal learning']


In [10]:
# Building a MinHash signature for each abstract

def make_minhash(shingle_set):
    m = MinHash(num_perm=num_perm)
    for shingle in shingle_set:
        m.update(shingle.encode("utf8"))
    return m

df_sample["minhash"] = df_sample["shingle_set"].apply(make_minhash)

# checking that one MinHash object was created for each abstract in the sample.
print("Rows in df_sample:", len(df_sample))
print("MinHash Signature", len(df_sample["minhash"]))

Rows in df_sample: 98618
MinHash Signature 98618


In [11]:
# Building the LSH index on top of the MinHash signatures

lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

with lsh.insertion_session() as session:
    for i in df_sample.index:
        session.insert(str(i), df_sample.loc[i, "minhash"])

# One example paper to test if the LSH index returns plausible candidate
# Result is the list of dataframe indixes that LSH considers potentially similar before exact Jaccard

query_index = df_sample.index[0]
candidate_indices = lsh.query(df_sample.loc[query_index, "minhash"])

print("Query index:", query_index)
print("LSH candidate indices:", candidate_indices)

Query index: 1790678
LSH candidate indices: ['1790678']


In [12]:
# Generating candidate similar pairs using LSH,checking for duplicates and self pairs
candidate_pairs = set()

for i in df_sample.index:
    result = lsh.query(df_sample.loc[i, "minhash"])
    for x in result:
        j = int(x)
        if i != j:
            candidate_pairs.add(tuple(sorted((i, j))))

print("Number of candidate pairs:", len(candidate_pairs))

Number of candidate pairs: 17


In [13]:
# I compute the exact Jaccard similarity and keep the final pairs.

def jaccard_similarity(set1, set2):
    common = set1.intersection(set2)
    total = set1.union(set2)
    score = len(common) / len(total)
    return score

similar_pairs = []

for i, j in candidate_pairs:
    set1 = df_sample.loc[i, "shingle_set"]
    set2 = df_sample.loc[j, "shingle_set"]
    score = jaccard_similarity(set1, set2)

    if score >= final_threshold:
        similar_pairs.append([
            df_sample.loc[i, "title"],
            df_sample.loc[j, "title"],
            score
        ])

similar_pairs_df = pd.DataFrame(
    similar_pairs,
    columns=["paper1_title", "paper2_title", "jaccard"]
)

similar_pairs_df = similar_pairs_df.sort_values("jaccard", ascending=False)
similar_pairs_df = similar_pairs_df.reset_index(drop=True)

print("Number of final similar pairs:", len(similar_pairs_df))
similar_pairs_df.head(15)

Number of final similar pairs: 12


,paper1_title,paper2_title,jaccard
0,"Machine Protection, Interlocks and Availability",Integration and (De-)installation,0.975904
1,"Fonctions constructibles exponentielles, trans...","Constructible exponential functions, motivic F...",0.794872
2,Electromagnetic Splittings and Light Quark Mas...,Electromagnetic Splittings and Light Quark Masses,0.782609
3,Real-Time Fair Resource Allocation in Distribu...,Multi-Path Alpha-Fair Resource Allocation at S...,0.767677
4,Evolution of the Magnetic Field in Accreting N...,Evolution of the Magnetic Field in Accreting N...,0.682927
5,Center vortex model for the infrared sector of...,Center vortex model for nonperturbative strong...,0.622951
6,AGILE observations of PSR B1509-58: is QED pho...,AGILE and Fermi Observations of the Soft Gamma...,0.617143
7,Enabling Conversational Behavior Reasoning Cap...,Conversational Behavior Modeling Foundation Mo...,0.612245
8,Phase diagram for the O(n) model with defects ...,"Cubic anisotropy created by defects of ""random...",0.568345
9,Production of W and Z bosons accompanied by je...,W and Z boson Productions in CMS at LHC startup,0.567901


In [15]:
# I select and save only the pairs with the highest similarity.
high_pairs_df = similar_pairs_df[similar_pairs_df["jaccard"] >= high_threshold].copy()
high_pairs_df = high_pairs_df.sort_values("jaccard", ascending=False)
high_pairs_df = high_pairs_df.reset_index(drop=True)

print("Number of high Jaccard pairs:", len(high_pairs_df))
high_pairs_df[["paper1_title", "paper2_title", "jaccard"]]

Number of high Jaccard pairs: 8


,paper1_title,paper2_title,jaccard
0,"Machine Protection, Interlocks and Availability",Integration and (De-)installation,0.975904
1,"Fonctions constructibles exponentielles, trans...","Constructible exponential functions, motivic F...",0.794872
2,Electromagnetic Splittings and Light Quark Mas...,Electromagnetic Splittings and Light Quark Masses,0.782609
3,Real-Time Fair Resource Allocation in Distribu...,Multi-Path Alpha-Fair Resource Allocation at S...,0.767677
4,Evolution of the Magnetic Field in Accreting N...,Evolution of the Magnetic Field in Accreting N...,0.682927
5,Center vortex model for the infrared sector of...,Center vortex model for nonperturbative strong...,0.622951
6,AGILE observations of PSR B1509-58: is QED pho...,AGILE and Fermi Observations of the Soft Gamma...,0.617143
7,Enabling Conversational Behavior Reasoning Cap...,Conversational Behavior Modeling Foundation Mo...,0.612245
